In [0]:
CREATE OR REPLACE VIEW gold.analistas.vw_producaomanaus AS (
-- ==============================================================================
-- PRODUÇÃO CONSOLIDADA MANAUS - GRANULARIDADE ADAPTATIVA (DATA + SKU + SÉRIE)
-- ==============================================================================
WITH 

-- 🛠️ Tabela de Capacidade Mensal Estática de Manaus/AM (vinda do DAX)
CAPACIDADE_MANAUS AS (
    SELECT Ano, Mes, Familia, Capacidade_Mes FROM (
        VALUES 
            -- 2025
            (2025, 1,  'TV', 14530),
            (2025, 2,  'TV', 14530),
            (2025, 3,  'TV', 16860),
            (2025, 4,  'TV', 21290),
            (2025, 5,  'TV', 17660),
            (2025, 6,  'TV', 17326),
            (2025, 7,  'TV', 17326),
            (2025, 8,  'TV', 28874),
            (2025, 9,  'TV', 28490),
            (2025, 10, 'TV', 25197),
            (2025, 11, 'TV', 25190),
            (2025, 12, 'TV', 25190),
            -- 2026
            (2026, 1,  'TV', 14416),
            (2026, 2,  'TV', 20412),
            (2026, 3,  'TV', 49054),
            (2026, 4,  'TV', 37673),
            (2026, 5,  'TV', 45440),
            (2026, 6,  'TV', 44603),
            (2026, 7,  'TV', 25947),
            (2026, 8,  'TV', 24085),
            (2026, 9,  'TV', 24085),
            (2026, 10, 'TV', 24085),
            (2026, 11, 'TV', 24085),
            (2026, 12, 'TV', 24085)
    ) AS t(Ano, Mes, Familia, Capacidade_Mes)
),

-- 🛠️ Busca a última localização de estoque do produto no catálogo de Manaus
ESTOQUE_LOCALIZACAO AS (
    SELECT 
        CodProduto, 
        MAX(CodLocal) AS CodLocalEstoque, 
        MAX(NomeLocal) AS DescricaoLocalEstoque
    FROM gold.sankhyaind.fato_estoque
    GROUP BY CodProduto
),

-- 🛠️ Filtros cadastrais de Produtos (Apenas TVs e Itens de Venda/Revenda)
SKU_ATRIBUTOS AS (
    SELECT
        P.CodProduto, 
        P.DescricaoProduto, 
        CASE WHEN P.Marca = 'PADRAO' THEN 'HQ' ELSE P.Marca END AS Fornecedor, 
        GP.NomeGrupoFamilia
    FROM gold.sankhyaind.dim_produtos P
    INNER JOIN gold.sankhyaind.dim_grupo_produtos GP ON GP.CodGrupoProduto = P.CodGrupoProduto
    WHERE P.UsadoComo IN ('Venda (fabricação própria)', 'Revenda')
      AND GP.NomeGrupoPai NOT IN ('COMPONENTES', 'PARTES E PECAS')
),

-- 🛠️ Quantidade Planejada da OP (Meta original de Engenharia)
PLANEJAMENTO_OP AS (
    SELECT 
        OrdemProducao,
        CodProdutoAcabado,
        MAX(CAST(QuantidadeAProduzir AS INT)) AS Qtd_Planejada_Item
    FROM gold.sankhyaind.fato_ordem_producao_item
    GROUP BY OrdemProducao, CodProdutoAcabado
),

-- 🛠️ Produção Real Adaptativa (Garante o histórico de 2025 onde não há série individual)
PRODUCAO_DIARIA AS (
    SELECT DISTINCT
        act.OrdemProducao,
        app.CodProdutoAcabado                             AS CodProd,
        CAST(ap.DataHoraApontamento AS DATE)              AS DataProducao,
        ap.DataHoraApontamento,
        -- Tratamento crucial: se a série for nula (comum em 2025), gera um identificador por lote/dia para não inibir a linha
        COALESCE(sp.SerieProdutoAcabado, CONCAT('LOTE-OP-', act.OrdemProducao, '-', CAST(ap.DataHoraApontamento AS DATE))) AS SerieProdutoAcabado,
        -- Se houver número de série real, contabiliza 1 unidade. Caso contrário, assume o volume total apontado no lote do dia
        CASE WHEN sp.SerieProdutoAcabado IS NOT NULL THEN 1 ELSE CAST(app.QuantidadeApontada AS INT) END AS Qtd_Produzida
    FROM gold.sankhyaind.fato_apontamento ap
    INNER JOIN gold.sankhyaind.fato_atividade_op act 
        ON ap.CodItemAtividade = act.CodItemAtividade
    INNER JOIN gold.sankhyaind.fato_apontamento_produto app 
        ON ap.CodApontamentoUnico = app.CodApontamentoUnico
    LEFT JOIN gold.sankhyaind.fato_produto_seriepa sp
        ON sp.CodApontamentoUnico = ap.CodApontamentoUnico
       AND sp.CodProdutoAcabado = app.CodProdutoAcabado
    LEFT JOIN gold.sankhyaind.dim_posto_trabalho dt 
        ON act.CodCentroTrabalho = dt.CodCentroTrabalho
    WHERE ap.DataHoraApontamento IS NOT NULL
      AND dt.CodCentroTrabalho IN (4, 8) -- Posto final de Embalagem
      AND CAST(ap.DataHoraApontamento AS DATE) BETWEEN '2025-01-01' AND CURRENT_DATE()
),

-- 🛠️ Reparos Consolidados por Série PA (Estratégia B: agregação completa sem expandir linhas)
REPAROS_CONSOLIDADO_MANAUS AS (
    SELECT
        m.SeriePA,
        COUNT(DISTINCT m.NroLancamento) AS Qtd_Eventos_Reparo,
        COUNT(rp.NroLancamentoProduto) AS Qtd_Componentes_Reparados,
        MAX(m.DataEntradaReparo) AS DataEntradaReparo,
        MAX(m.DataFimReparo) AS DataFimReparo,
        CONCAT_WS(' | ', COLLECT_SET(NULLIF(TRIM(m.Defeito), ''))) AS Defeitos,
        CONCAT_WS(' | ', COLLECT_SET(NULLIF(TRIM(rp.Causa), ''))) AS Causas,
        CONCAT_WS(' | ', COLLECT_SET(NULLIF(TRIM(rp.Acao), ''))) AS Acoes,
        CONCAT_WS(' | ', COLLECT_SET(CASE WHEN rp.SerieProdutoMP <> 'N/A' THEN rp.SerieProdutoMP END)) AS Series_Componente_Antigo,
        CONCAT_WS(' | ', COLLECT_SET(CASE WHEN rp.SerieProdutoNovaMP <> 'N/A' THEN rp.SerieProdutoNovaMP END)) AS Series_Componente_Novo,
        CONCAT_WS(' | ', COLLECT_SET(NULLIF(TRIM(rp.OrigemFornecedor), ''))) AS Fornecedores_Origem
    FROM gold.sankhyaind.fato_motivo_reparo_producao m
    LEFT JOIN gold.sankhyaind.fato_reparo_produto rp ON m.NroLancamento = rp.NroLancamento
    GROUP BY m.SeriePA
)

-- ==============================================================================
-- CONSOLIDADO FINAL COM REPAROS AGREGADOS E SALDO AJUSTADO
-- ==============================================================================
SELECT 
    EST.CodLocalEstoque,
    EST.DescricaoLocalEstoque,
    P.DataProducao,
    YEAR(P.DataProducao)                                AS Ano,
    MONTH(P.DataProducao)                               AS Mes,
    WEEKOFYEAR(P.DataProducao)                          AS Semana,
    P.DataHoraApontamento,
    P.OrdemProducao                                     AS OP,
    P.CodProd,
    SKU.DescricaoProduto,
    SKU.Fornecedor,
    SKU.NomeGrupoFamilia,
    'TV'                                                AS Familia,
    'MANAUS/AM'                                         AS Planta,
    'PRODUZIDO'                                         AS Situacao,
    
    -- Volumes de Produção e Saldos Corrigidos
    P.SerieProdutoAcabado,
    P.Qtd_Produzida,
    -- Pro-rata: distribui Qtd_Planejada proporcionalmente à produção de cada mês
    -- (corrige OFR > 100% causado por OPs que cruzam meses)
    CASE WHEN ROW_NUMBER() OVER(PARTITION BY P.OrdemProducao, P.CodProd, YEAR(P.DataProducao), MONTH(P.DataProducao) ORDER BY P.DataProducao, P.SerieProdutoAcabado) = 1
         THEN ROUND(
             COALESCE(PLAN.Qtd_Planejada_Item, 0)
             * SUM(P.Qtd_Produzida) OVER(PARTITION BY P.OrdemProducao, P.CodProd, YEAR(P.DataProducao), MONTH(P.DataProducao))
             / NULLIF(SUM(P.Qtd_Produzida) OVER(PARTITION BY P.OrdemProducao, P.CodProd), 0)
         , 0)
         ELSE 0
    END AS Qtd_Planejada,
    COALESCE(PLAN.Qtd_Planejada_Item, 0) 
        - SUM(P.Qtd_Produzida) OVER(PARTITION BY P.OrdemProducao, P.CodProd ORDER BY P.DataProducao, P.SerieProdutoAcabado) AS Saldo,

    -- 🎯 CAPACIDADE PRODUTIVA MENSAL (Injetada no 1º registro da 1ª data com produção do mês)
    CASE 
        WHEN ROW_NUMBER() OVER(
            PARTITION BY YEAR(P.DataProducao), MONTH(P.DataProducao)
            ORDER BY P.DataProducao ASC, P.DataHoraApontamento ASC, P.OrdemProducao, P.SerieProdutoAcabado
        ) = 1 
        THEN COALESCE(CAP.Capacidade_Mes, 0)
        ELSE 0 
    END AS Capacidade_Produtiva,

    -- 🔧 DADOS DE REPARO (agregados por Série PA)
    COALESCE(REP.Qtd_Eventos_Reparo, 0) AS Qtd_Eventos_Reparo,
    COALESCE(REP.Qtd_Componentes_Reparados, 0) AS Qtd_Componentes_Reparados,
    REP.DataEntradaReparo,
    REP.DataFimReparo,
    COALESCE(REP.Defeitos, 'Sem Reparo') AS Defeitos,
    COALESCE(REP.Causas, 'Sem Reparo') AS Causas,
    COALESCE(REP.Acoes, 'Sem Reparo') AS Acoes,
    COALESCE(REP.Series_Componente_Antigo, 'Sem Reparo') AS Series_Componente_Antigo,
    COALESCE(REP.Series_Componente_Novo, 'Sem Reparo') AS Series_Componente_Novo,
    COALESCE(REP.Fornecedores_Origem, 'Sem Reparo') AS Fornecedores_Origem

FROM PRODUCAO_DIARIA P
INNER JOIN SKU_ATRIBUTOS SKU 
    ON P.CodProd = SKU.CodProduto
LEFT JOIN PLANEJAMENTO_OP PLAN 
    ON P.OrdemProducao = PLAN.OrdemProducao AND P.CodProd = PLAN.CodProdutoAcabado
LEFT JOIN ESTOQUE_LOCALIZACAO EST 
    ON P.CodProd = EST.CodProduto
LEFT JOIN CAPACIDADE_MANAUS CAP
    ON YEAR(P.DataProducao) = CAP.Ano
   AND MONTH(P.DataProducao) = CAP.Mes
   AND 'TV' = CAP.Familia
LEFT JOIN REPAROS_CONSOLIDADO_MANAUS REP
    ON P.SerieProdutoAcabado = REP.SeriePA

ORDER BY 
    P.DataHoraApontamento DESC, 
    P.OrdemProducao, 
    P.CodProd
);